## Evaluation

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_squared_error
import numpy as np
import seaborn as sns

Calculate average of a measurement, will be used in the next function

In [3]:
def average(data, measurement):
    avg = data[measurement].mean()
    return avg

Calculate error margin with mean square error. The percentage is calculated to get a comparable value. The error is divided by average of the measurement to get the percentage error. The mean square error, root mean square error and the percentage are added to a dataframe. You can chose to have a limitation of the error percetage by adding a redflag percent, the function will then return a dataframe with the error margin and a dataframe with only the measurement that did not meet the requierment. If a redflag percentage is not added then only the error margin dataframe will be returned.

In [4]:
def calc_error_margin(prediction, test, redflag_percent=False):
    test = test.iloc[:len(prediction),:]
    error_margin= pd.DataFrame()
    red_flags= pd.DataFrame()
    ansur=pd.read_csv('../data/processed/ANSURIImalefemale.csv')
    for m in list(prediction.columns[:]):
        mse= mean_squared_error(prediction[m], test[m])
        rmse=np.sqrt(mse)
        avg = average(ansur,m)
        error_margin[m]= {'mse': mse, 'rmse': rmse, '%' : (rmse/avg)*100 }
        if redflag_percent!= False and (rmse/avg)*100>redflag_percent:
            red_flags[m]=error_margin[m]
    if redflag_percent!=False:
        return error_margin, red_flags
    return error_margin

The function below plots the actual value in realation to the predicted value. It is a scatter plot where hue is based on gender. The gender 0 represents females while 1 represent males.

In [5]:
def plot_errors(error_measurments, prediction, test):
   y_test=test.iloc[:,1:94].drop('weightkg',axis=1).drop('stature',axis=1)
   for r in error_measurments:
      sns.scatterplot(x = prediction[r], y = y_test[r], hue=test["Gender"])
      plt.plot([min(prediction[r]), max(prediction[r])], [min(prediction[r]), max(prediction[r])],linewidth=3, c="red")
      plt.title(r)
      plt.xlabel("Predicted")
      plt.ylabel("Actual")
      plt.show()


Here we are testing the errors of different models.

In [13]:
test=pd.read_csv('../data/processed/ANSURIItest.csv')
test_pred=pd.read_csv('../output/anthro_results/predict_all_test_xgboost.csv')
error, r = calc_error_margin(test_pred,test,7)
t=error.T
t.mean()

mse     382.122518
rmse     16.886951
%         4.140638
dtype: float64

In [7]:
test=pd.read_csv('../data/processed/ANSURIItest.csv')
predict_test_bambi_c=pd.read_csv('../output/anthro_results/predict_test_bambi_c.csv')
predict_test_bambi=pd.read_csv('../output/anthro_results/predict_test_bambi.csv')
predict_test_xgboost=pd.read_csv('../output/anthro_results/predict_test_xgboost.csv')


In [8]:
error_b_c , r_b_c =calc_error_margin(predict_test_bambi_c, test, 7)
error_b, r_b =calc_error_margin(predict_test_bambi, test, 7)
error_xgb, r_xgb =calc_error_margin(predict_test_xgboost, test, 7)

In [58]:
error_b_c.to_csv('../output/anthro_results/errors/error_b_c.csv', index=False)
error_b.to_csv('../output/anthro_results/errors/error_b.csv', index=False)
error_xgb.to_csv('../output/anthro_results/errors/error_xgb.csv', index=False)

In [79]:
r_b_c.to_csv('../output/anthro_results/errors/r_b_c.csv', index=False)
r_b.to_csv('../output/anthro_results/errors/r_b.csv', index=False)
r_xgb.to_csv('../output/anthro_results/errors/r_xgb.csv', index=False)

In [63]:
error_b_c.T['%'].mean()

4.159631261945237

In [64]:
error_xgb.T['%'].mean()

4.187657064855659

In [69]:
top5bc=error_b_c[['buttockcircumference',
'chestcircumference',
'waistcircumference',
'headcircumference',
'thighcircumference', 
]]#'calflink','waistbacklength','thighlink','hipbreadth','calfcircumference'

top5xgb=error_xgb[['buttockcircumference',
'chestcircumference',
'waistcircumference',
'headcircumference',
'thighcircumference', 
]]

In [72]:
top5xgb

,buttockcircumference,chestcircumference,waistcircumference,headcircumference,thighcircumference
mse,777.261202,1478.695160,2230.542218,228.788421,412.497213
rmse,27.879405,38.453806,47.228617,15.125754,20.310027
%,2.733048,3.762239,5.164411,2.653477,3.264235


In [70]:
top5bc.to_csv('../output/anthro_results/errors/top5bc.csv', index=False)
top5xgb.to_csv('../output/anthro_results/errors/top5xgb.csv', index=False)